# CardioScore Validation 04 — Robustness & Reporting

Secondary analysis only. It must never overwrite the locked primary result.

In [ ]:

import sys, subprocess, json, hashlib, zipfile, tarfile
from pathlib import Path
PIN = "869150cd5fb5ccf155fb066258404bd4df163ade"
REPO = "Virelion-Biotech/Virelion-CardioScore"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"git+https://github.com/{REPO}.git@{PIN}"], check=True)
print("Installed pinned CardioScore:", PIN)


In [ ]:

assert Path("/content/cardioscore_validation/results/locked_external_validation.json").exists(), "Run notebook 03 first."
locked = json.loads(Path("/content/cardioscore_validation/results/locked_external_validation.json").read_text())
compound_scores = pd.DataFrame(locked["compound_scores"])
features = pd.read_csv("/content/cardioscore_validation/locked_input/canonical_features.csv")
reference = pd.read_csv("/content/cardioscore_validation/locked_input/reference.csv")


In [ ]:

from virelion_cardioscore.analysis.pipeline import CardioScorePipeline
base = CardioScorePipeline.from_defaults()
base_result = base.run(features)
baseline = base_result.summary_table[["compound","cardioscore","risk_class"]].copy()

endpoint_path = Path("/content/cardioscore_validation/endpoint_sensitivity.yaml")
base_endpoint_cfg = yaml.safe_load((Path(__import__("virelion_cardioscore").__file__).resolve().parent/"config"/"cipa_endpoints.yaml").read_text())

def score_with_weights(weights):
    cfg = json.loads(json.dumps(base.config, default=str))
    ep = json.loads(json.dumps(base_endpoint_cfg))
    for k,w in weights.items(): ep["endpoints"][k]["weight"] = float(w)
    total = sum(float(v["weight"]) for v in ep["endpoints"].values())
    for v in ep["endpoints"].values(): v["weight"] = float(v["weight"])/total
    ep_path = Path("/content/cardioscore_validation")/("endpoints_"+str(abs(hash(tuple(sorted(weights.items())))))+".yaml")
    ep_path.write_text(yaml.safe_dump(ep, sort_keys=False))
    cfg["scoring"]["endpoint_config"] = str(ep_path)
    p = CardioScorePipeline(cfg)
    return p.run(features).summary_table[["compound","cardioscore","risk_class"]].copy()

scenarios = {
    "baseline": {k: float(v["weight"]) for k,v in base_endpoint_cfg["endpoints"].items()},
    "fpd_only": {"fpd_change_pct":1,"beat_rate_change_pct":0,"amplitude_change_pct":0,"stv_increase":0,"triangulation_proxy":0},
    "no_fpd": {"fpd_change_pct":0,"beat_rate_change_pct":0.30,"amplitude_change_pct":0.20,"stv_increase":0.30,"triangulation_proxy":0.20},
}
rows=[]
for name, weights in scenarios.items():
    if name=="baseline": cur=baseline
    else: cur=score_with_weights(weights)
    cur=cur.rename(columns={"cardioscore":"score_"+name,"risk_class":"risk_"+name})
    rows.append(cur)
sens=rows[0]
for cur in rows[1:]: sens=sens.merge(cur,on="compound",how="inner",validate="one_to_one")


In [ ]:

rows=[]
for name, factor in [("thresholds_minus10",0.90),("thresholds_plus10",1.10)]:
    cfg = json.loads(json.dumps(base.config, default=str))
    ep = json.loads(json.dumps(base_endpoint_cfg))
    for v in ep["endpoints"].values(): v["effect_threshold"] = float(v["effect_threshold"]) * factor
    ep_path = Path("/content/cardioscore_validation")/(name+".yaml")
    ep_path.write_text(yaml.safe_dump(ep, sort_keys=False))
    cfg["scoring"]["endpoint_config"] = str(ep_path)
    r = CardioScorePipeline(cfg).run(features).summary_table[["compound","cardioscore","risk_class"]]
    r.columns=["compound",name+"_score",name+"_risk"]
    rows.append(r)
for r in rows: sens=sens.merge(r,on="compound",how="inner",validate="one_to_one")
Path("/content/cardioscore_validation/results").mkdir(parents=True,exist_ok=True)
sens.to_csv("/content/cardioscore_validation/results/secondary_sensitivity.csv",index=False)
print(sens.to_string(index=False))


In [ ]:

p = CardioScorePipeline.from_defaults()
r = p.run(features)
qc = pd.DataFrame({"qc_log": r.qc_log})
qc.to_csv("/content/cardioscore_validation/results/qc_log.csv", index=False)
summary = {
    "n_input_rows": int(len(features)),
    "n_scored_compounds": int(len(r.summary_table)),
    "n_reference_compounds": int(len(reference)),
    "n_excluded_reference_compounds": int(len(set(reference["compound"]) - set(r.summary_table["compound"])))
}
Path("/content/cardioscore_validation/results/qc_summary.json").write_text(json.dumps(summary,indent=2)+"\n")
print(json.dumps(summary,indent=2))


### Reporting rule
Report the locked primary metrics first, then sensitivity/ablation results and QC/dropout. Do not select a secondary scenario because it produces a more favorable classification.